<a href="https://colab.research.google.com/github/umang0015/Ai-DataScience/blob/main/NLP/12AugSession_1ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Models/SPAM text - SPAM text.csv')

In [ ]:
#next apply preprocessing onto the data
# import necessary libraries first

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
import re
import string
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
#ml

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
#now apply preprocessing
# build a preprocessing function

def text_process(text):
  # convert text to lower
  text = text.lower()
  #remove the number using re
  text = re.sub(r'\d+', '', text)
  # remoce punctuation
  text = text.translate(str.maketrans('', '', string.punctuation))
  # remove white space
  text = text.strip()
  #now tokenize the text
  tokens=text.split()
  #remove stopwords
  final_tokens = [word for word in tokens if word not in stopwords.words('english')]
  #stemming
  ps = PorterStemmer()
  final_tokens = [ps.stem(word) for word in final_tokens]
  # join the tokens
  text = ' '.join(final_tokens)
  return text

In [ ]:
df['cleaned message']=df['Message'].apply(text_process)
df.head(2)

,Category,Message,cleaned message
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif u oni


In [ ]:
# #loaded the data
# # import library
# split into x and y
# definded preprocess function
# cleaned x
# install gensim
# import w2v apply onto cleaned x
# split into train and test set

In [ ]:
!pip install gensim
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 55.0 MB/s eta 0:00:00


In [ ]:
df['tokens']=df['cleaned message'].apply(lambda x:x.split())
df['tokens'].head()

,tokens
0,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,"[ok, lar, joke, wif, u, oni]"
2,"[free, entri, wkli, comp, win, fa, cup, final,..."
3,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,"[nah, dont, think, goe, usf, live, around, tho..."


In [ ]:
w2v_model = Word2Vec(df['tokens'],vector_size=100,window=5,min_count=1,workers=4)

In [ ]:
#word2vec gives one vector per word
#we need it for vector for a message(each row)
#define function which converts a vector to a word to vector for a message
def get_doc_vector(tokens,model):
  vectors=[model.wv[word] for word in tokens if word in model.wv]
  if vectors:
    return np.mean(vectors,axis=0)
  else:
    return np.zeros(model.wv.vector_size)

In [ ]:
#call the function
x3 = np.array([get_doc_vector(tokens,w2v_model)for tokens in df['tokens']])
y3 = df['Category']

In [ ]:
#split train test
X_train,X_test,Y_train,Y_test = train_test_split(x3,y3,test_size=0.2,random_state=42)

In [ ]:
# ANN PART , TRAIN ABOVE DATASET USING ANN
# LABEL ENCODE THE TARGET COLUMN
from sklearn.preprocessing import LabelEncoder
LE = LabelEncoder()
y_encoded = LE.fit_transform(y3)
y_encoded

array([0, 0, 1, ..., 0, 0, 0])

In [ ]:
#apply standardization into inputs

In [ ]:

#split train test
X_train,X_test,Y_train,Y_test = train_test_split(x3,y_encoded,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

(4457, 100)
(1115, 100)
(4457,)
(1115,)


In [ ]:
#import ANN libraries and build model.
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

In [ ]:
# define the architecture
model = Sequential()
model.add(Dense(units=128,input_shape=(X_train.shape[1],),activation='relu'))
model.add(Dense(units=64,activation='relu'))
model.add(Dense(units=1,activation='sigmoid'))
model.summary()
model.compile(optimizer=Adam(learning_rate=0.001),loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,249 (83.00 KB)

 Trainable params: 21,249 (83.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# train the model
history = model.fit(X_train,Y_train,epochs=10,batch_size=32,validation_data=(X_test,Y_test))

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8658 - loss: 0.4198 - val_accuracy: 0.8664 - val_loss: 0.3963
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8658 - loss: 0.3824 - val_accuracy: 0.8664 - val_loss: 0.3777
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8658 - loss: 0.3533 - val_accuracy: 0.8664 - val_loss: 0.3424
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8705 - loss: 0.3175 - val_accuracy: 0.8816 - val_loss: 0.2816
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8932 - loss: 0.2645 - val_accuracy: 0.8897 - val_loss: 0.2783
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9147 - loss: 0.2342 - val_accuracy: 0.8915 - val_loss: 0.3277
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9204 - loss: 0.2195 - val_accuracy: 0.9202 - val_loss: 0.2044
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9287 - loss: 0.1980 - val_accuracy

In [ ]:
# print the confusion matrix
y_pred = model.predict(X_test)
y_pred = np.where(y_pred>0.5,1,0)
cm = confusion_matrix(Y_test,y_pred)
print(cm)

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
[[905  61]
 [ 28 121]]


In [ ]:
# to train text data on rnn or lstms the data has to be converted into sequence
# as we have vector for each message after conversaion it will become
# sequence of vectors


In [ ]:
# convert input data to sequence
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

embedding_dim=w2v_model.wv.vector_size
max_length=50
X_sequence=[]# here we add word vectors
for tokens in df['tokens']:
  sequence=[w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
  X_sequence.append(sequence)
len(X_sequence)
x3.shape

(5572, 100)

In [ ]:
#pad the sequence
X_rnn = pad_sequences(X_sequence,maxlen=max_length,padding='post',truncating='post')
X_rnn.shape

(5572, 50, 100)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X_rnn,y_encoded,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(4457, 50, 100)
(1115, 50, 100)
(4457,)
(1115,)


In [ ]:
# build simple layer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,LSTM,SimpleRNN

rnn_model=Sequential()
rnn_model.add(SimpleRNN(units=64,input_shape=(max_length,embedding_dim),activation='tanh',return_sequences=False))
rnn_model.add(Dropout(0.2)) # dropout with 20%
rnn_model.add(Dense(units=1,activation='sigmoid'))
rnn_model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',metrics=['accuracy'])
rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        10,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,625 (41.50 KB)

 Trainable params: 10,625 (41.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# train the rnn model
history_rnn=rnn_model.fit(X_train,y_train,epochs=10,batch_size=32,validation_split=0.2)

Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.8673 - loss: 0.4113 - val_accuracy: 0.8576 - val_loss: 0.4102
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8679 - loss: 0.4008 - val_accuracy: 0.8576 - val_loss: 0.4103
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8679 - loss: 0.3993 - val_accuracy: 0.8576 - val_loss: 0.4095
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8676 - loss: 0.3933 - val_accuracy: 0.8576 - val_loss: 0.4026
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.8659 - loss: 0.3855 - val_accuracy: 0.8576 - val_loss: 0.3704
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.8626 - loss: 0.3515 - val_accuracy: 0.8576 - val_loss: 0.3300
Epoch 7/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8676 - loss: 0.3433 - val_accuracy: 0.8576 - val_loss: 0.3668
Epoch 8/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8656 - loss: 0.3327 - val_accu

In [ ]:
# LSTM for classification
lstm_model = Sequential()
lstm_model.add(LSTM(units=64,input_shape=(max_length,embedding_dim),activation='tanh',return_sequences=False))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(units=1,activation='sigmoid'))
lstm_model.compile(optimizer=Adam(learning_rate=0.001),loss='binary_crossentropy',metrics=['accuracy'])
lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,305 (165.25 KB)

 Trainable params: 42,305 (165.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = lstm_model.fit(X_train,y_train,epochs=10,batch_size=32,validation_split=0.2)

Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.8612 - loss: 0.4066 - val_accuracy: 0.8565 - val_loss: 0.3397
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.8656 - loss: 0.3239 - val_accuracy: 0.8543 - val_loss: 0.3148
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.8640 - loss: 0.3136 - val_accuracy: 0.8722 - val_loss: 0.3115
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.8642 - loss: 0.3064 - val_accuracy: 0.8587 - val_loss: 0.2984
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.8684 - loss: 0.2990 - val_accuracy: 0.8711 - val_loss: 0.2966
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.8783 - loss: 0.2913 - val_accuracy: 0.8677 - val_loss: 0.3013
Epoch 7/10


In [ ]:
#add early stopping and retrain the model
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)

history = lstm_model.fit(X_train,y_train,epochs=25,batch_size=32,callbacks=[early_stopping],validation_split=0.2)